# Notebook 18 — Residual Stream Direction Hypothesis: Where Does the Damage Land?
**CS 590NN | Amogh | Apr 25 — final experiment, bridges editing and interpretability**

## What we're testing

Every previous experiment measured damage in **parameter space** (weight deltas, circuit overlaps, cosine alignment of weight changes). NB17 falsified the parameter-space cosine hypothesis. The conclusion was that damage probably operates in *functional* not *parametric* space.

**NB18 hypothesis:** Edit B's optimizer step damages A by writing into the **residual stream direction** that A's installation created at the final layer, regardless of which MLPs are physically modified.

In other words: when we feed A's prompt to the network after edit A, the residual stream at the final layer points in some direction `h_A` that encodes "predict target_new_A". Edit B's parameter changes alter this direction by some `delta_h`. The damage to A is proportional to **how much `delta_h` opposes `h_A`** — i.e., how negatively `delta_h` projects onto `h_A`.

## Why this is genuinely novel

- Editing literature: measures input/output (probability changes) or parameter-space deltas.
- Mechanistic interpretability literature: measures activations and circuits but rarely as outcomes of editing.
- **No paper measures residual stream direction induced by an edit and tracks how subsequent edits perturb that direction.**

This bridges the two literatures and gives us a **mechanistic localization of damage** — in the residual stream, not just "in the weights."

## Predictions

| Pattern | Conclusion |
|---|---|
| Strong negative projection correlates with A_displaced (rho < -0.4) | Damage is residual-direction-aligned. Mechanistic localization confirmed. |
| Magnitude of `delta_h` correlates without sign | Damage is generic perturbation, direction doesn't matter. |
| Null | Damage operates at even finer granularity (per-token, per-position) |

## Compute

~20 min A100. Same 3-phase protocol as NB17 but with residual stream hooks instead of weight snapshots.

### 18.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 18.1 Imports + auto-fetch v3 pair indices

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb18'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'
v3_data = requests.get(V3_URL, timeout=60).json()
v3_ok = [r for r in v3_data['rows'] if r.get('status') == 'ok']
seen = set()
PAIR_INDICES = []
for r in v3_ok:
    pid = r['pair_idx']
    if pid in seen: continue
    seen.add(pid)
    PAIR_INDICES.append((r['A_idx'], r['B_idx']))
    if len(PAIR_INDICES) >= 10: break
print(f'Selected {len(PAIR_INDICES)} pairs')

### 18.2 Load model + samples

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token

@dataclass
class EditSample:
    idx: int; prompt: str; subject: str
    target_new: str; target_true: str

raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      subject=rr['subject'],
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

pairs = [{'A': make_sample(a), 'B': make_sample(b)} for a, b in PAIR_INDICES]
print(f'Loaded {len(pairs)} pairs, d_model={model.cfg.d_model}, n_layers={model.cfg.n_layers}')

### 18.3 ROME-trace + helpers (same as before)

In [ ]:
def rome_trace_top_layers(model, sample, k=5):
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0].item()
    tokens = model.to_tokens(sample.prompt)
    n_tok = tokens.shape[1]
    if n_tok <= 2: return list(range(min(k, model.cfg.n_layers)))
    subj_pos = list(range(1, n_tok-1))
    corrupt = tokens.clone()
    corrupt[0, subj_pos] = torch.randint(1000, model.cfg.d_vocab-1, (len(subj_pos),), device=tokens.device)
    with torch.no_grad():
        cl, cc = model.run_with_cache(tokens)
        kl, _ = model.run_with_cache(corrupt)
    clean_score = (cl[0,-1,true_id] - cl[0,-1,new_id]).item()
    corrupt_score = (kl[0,-1,true_id] - kl[0,-1,new_id]).item()
    eff = max(abs(clean_score - corrupt_score), 0.5)
    effects = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        if hn not in cc: continue
        ca = cc[hn].clone()
        def hk(v, hook): return ca
        with torch.no_grad():
            patched = model.run_with_hooks(corrupt, fwd_hooks=[(hn, hk)])
        recovery = (patched[0,-1,true_id] - patched[0,-1,new_id]).item() - corrupt_score
        effects.append((L, abs(recovery)/eff))
    effects.sort(key=lambda x: -x[1])
    del cc; torch.cuda.empty_cache()
    return [L for L, _ in effects[:k]]

NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
    'A week contains seven', 'The primary colors are red, blue, and',
    'Humans have two lungs and one', 'A triangle has three',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            t = model.to_tokens(p)
            ref_lp = torch.log_softmax(model(t)[0,-1,:], dim=-1).detach().clone()
            cache.append((t, ref_lp))
    return cache

def kl_against(model, ref_cache):
    total = 0.0
    for t, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(t)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def edit_normal(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache):
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    if not params:
        for L in range(model.cfg.n_layers):
            params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return float(torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id])

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

print('Helpers ready.')

### 18.4 Residual stream extraction

In [ ]:
def get_residual_at_final_layer(model, sample):
    """
    Extract residual stream at the final layer for the last token position.
    Returns: tensor of shape (d_model,)
    """
    model.eval()
    tokens = model.to_tokens(sample.prompt)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
    # Use post-final-block residual stream at last token position
    final_layer = model.cfg.n_layers - 1
    h_name = f'blocks.{final_layer}.hook_resid_post'
    if h_name not in cache:
        h_name = f'blocks.{final_layer}.hook_resid_pre'
    h = cache[h_name][0, -1, :].detach().clone().cpu()
    del cache; torch.cuda.empty_cache()
    return h

def get_residual_per_layer(model, sample):
    """Extract residual stream at every layer for the last token, returns (n_layers, d_model)."""
    model.eval()
    tokens = model.to_tokens(sample.prompt)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
    layers = []
    for L in range(model.cfg.n_layers):
        h_name = f'blocks.{L}.hook_resid_post'
        if h_name not in cache:
            h_name = f'blocks.{L}.hook_resid_pre'
        layers.append(cache[h_name][0, -1, :].detach().clone().cpu())
    del cache; torch.cuda.empty_cache()
    return torch.stack(layers)

print('Residual extraction ready.')

### 18.5 The 3-phase protocol with residual measurements

In [ ]:
N_STEPS_A = 5
N_STEPS_B = 3
BETA_KL = 0.1
LR = 5e-3

ref_cache = cache_ref(model, NEUTRAL)

all_samples = {s.idx: s for p in pairs for s in (p['A'], p['B'])}
print(f'ROME-trace for {len(all_samples)} samples...')
circuits = {idx: rome_trace_top_layers(model, s, k=5) for idx, s in all_samples.items()}
print('Done.')

print('\nSnapshotting original weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()

ROWS_FILE = RESULTS_DIR / f'residual_stream_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
rows = []
started = datetime.now()

for p_idx, p in enumerate(pairs):
    A, B = p['A'], p['B']
    layers_A = circuits[A.idx]
    layers_B = circuits[B.idx]
    
    try:
        # ===== Phase 0: Original residual on A's prompt =====
        restore(model, original_state)
        h_A_orig = get_residual_at_final_layer(model, A)
        h_A_orig_per_layer = get_residual_per_layer(model, A)
        
        # ===== Phase 1: Edit A → measure h_A_postA on A's prompt =====
        edit_normal(model, A, layers_A, n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        h_A_postA = get_residual_at_final_layer(model, A)
        h_A_postA_per_layer = get_residual_per_layer(model, A)
        p_A_postA = measure_p_new(model, A)
        # The edit's contribution to A's residual
        h_A_edit_contribution = h_A_postA - h_A_orig
        
        # ===== Phase 2: Restore, edit B solo, measure h_A_after_B_solo on A's prompt =====
        restore(model, original_state)
        edit_normal(model, B, layers_B, n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        h_A_after_B_solo = get_residual_at_final_layer(model, A)  # still A's prompt
        h_A_after_B_solo_per_layer = get_residual_per_layer(model, A)
        # B's perturbation to A's residual
        delta_h_from_B = h_A_after_B_solo - h_A_orig
        
        # ===== Compute key quantities =====
        # 1. cosine between B's perturbation of A's residual, and A's edit contribution
        cos_perturbation = float(torch.nn.functional.cosine_similarity(
            delta_h_from_B.unsqueeze(0), h_A_edit_contribution.unsqueeze(0)).item())
        
        # 2. projection of delta_h_from_B onto h_A_edit_contribution direction
        # If positive: B reinforces A's direction. If negative: B opposes it.
        norm_h_edit = float(h_A_edit_contribution.norm().item()) + 1e-12
        projection_signed = float((delta_h_from_B @ h_A_edit_contribution).item()) / norm_h_edit
        
        # 3. magnitude of B's perturbation on A's residual
        mag_delta_h = float(delta_h_from_B.norm().item())
        mag_h_edit = float(h_A_edit_contribution.norm().item())
        
        # ===== Per-layer projections =====
        per_layer_proj = []
        for L in range(model.cfg.n_layers):
            h_edit_L = h_A_postA_per_layer[L] - h_A_orig_per_layer[L]
            delta_L = h_A_after_B_solo_per_layer[L] - h_A_orig_per_layer[L]
            n = float(h_edit_L.norm().item()) + 1e-12
            proj = float((delta_L @ h_edit_L).item()) / n
            per_layer_proj.append(proj)
        
        # ===== Phase 3: Sequential edit A→B, measure A_displaced =====
        restore(model, original_state)
        edit_normal(model, A, layers_A, n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        edit_normal(model, B, layers_B, n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        p_A_postAB = measure_p_new(model, A)
        p_B_postAB = measure_p_new(model, B)
        A_displaced = max(0, p_A_postA - p_A_postAB)
        
        row = {
            'pair_idx': p_idx, 'A_idx': A.idx, 'B_idx': B.idx,
            'cos_perturbation_vs_edit': cos_perturbation,
            'projection_signed': projection_signed,
            'mag_delta_h_from_B': mag_delta_h,
            'mag_h_edit_contribution': mag_h_edit,
            'p_A_postA': p_A_postA,
            'p_A_postAB': p_A_postAB,
            'p_B_postAB': p_B_postAB,
            'A_displaced': A_displaced,
            'per_layer_projections': per_layer_proj,
            'status': 'ok',
        }
        rows.append(row)
        print(f'[{p_idx+1}/{len(pairs)}] pair {p_idx}: '
              f'cos={cos_perturbation:+.3f}  proj={projection_signed:+.2f}  '
              f'|δh|={mag_delta_h:.2f}  |h_edit|={mag_h_edit:.2f}  A_disp={A_displaced:.3f}')
        with open(ROWS_FILE, 'w') as f: json.dump({'rows': rows}, f, indent=2)
    except Exception as e:
        print(f'[{p_idx+1}/{len(pairs)}] FAILED: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
        rows.append({'pair_idx': p_idx, 'status': 'failed', 'error': str(e)[:200]})
        torch.cuda.empty_cache()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'\nTotal runtime: {elapsed:.1f} min')
restore(model, original_state)

### 18.6 Test the hypothesis

In [ ]:
df = pd.DataFrame([r for r in rows if r.get('status') == 'ok'])
print(f'OK rows: {len(df)}')
print('\nDescriptive stats:')
print(df[['cos_perturbation_vs_edit', 'projection_signed', 'mag_delta_h_from_B',
         'mag_h_edit_contribution', 'A_displaced']].describe().round(3))

print('\n=== HEADLINE TEST: residual stream metrics vs A_displaced ===')

print('\n--- Cosine: angle between B_perturbation and A_edit_contribution in residual space ---')
rho_cos, p_cos = stats.spearmanr(df['cos_perturbation_vs_edit'], df['A_displaced'])
sig = '***' if p_cos<0.001 else ('**' if p_cos<0.01 else ('*' if p_cos<0.05 else 'ns'))
print(f'  Spearman rho = {rho_cos:+.3f}  p = {p_cos:.4f}  {sig}')

print('\n--- Signed projection (negative = B opposes A direction) ---')
rho_proj, p_proj = stats.spearmanr(df['projection_signed'], df['A_displaced'])
sig = '***' if p_proj<0.001 else ('**' if p_proj<0.01 else ('*' if p_proj<0.05 else 'ns'))
print(f'  Spearman rho = {rho_proj:+.3f}  p = {p_proj:.4f}  {sig}')
print('  [If rho is NEGATIVE: more negative projection (B opposes A) → more damage. Hypothesis supported.]')

print('\n--- Magnitude of B perturbation on A residual ---')
rho_mag, p_mag = stats.spearmanr(df['mag_delta_h_from_B'], df['A_displaced'])
sig = '***' if p_mag<0.001 else ('**' if p_mag<0.01 else ('*' if p_mag<0.05 else 'ns'))
print(f'  Spearman rho = {rho_mag:+.3f}  p = {p_mag:.4f}  {sig}')

# Multivariate
print('\n=== Multivariate: A_displaced ~ projection + cos + magnitude ===')
from numpy.linalg import lstsq
cols = ['projection_signed', 'cos_perturbation_vs_edit', 'mag_delta_h_from_B']
sub = df.dropna(subset=cols + ['A_displaced'])
X = np.column_stack([np.ones(len(sub)), sub[cols].values])
y = sub['A_displaced'].values
beta, _, _, _ = lstsq(X, y, rcond=None)
yp = X @ beta
r2 = 1 - ((y-yp)**2).sum() / ((y-y.mean())**2).sum()
coefs = ', '.join([f'{c.split("_")[0]}={beta[i+1]:+.4f}' for i, c in enumerate(cols)])
print(f'  R²={r2:.3f}  intercept={beta[0]:+.3f}')
print(f'  {coefs}')

# Per-layer analysis: which layers' projections matter?
print('\n=== Per-layer projection vs A_displaced ===')
n_layers = len(df.iloc[0]['per_layer_projections'])
print(f'  Computing correlation per layer (n_layers={n_layers}):')
layer_rhos = []
for L in range(n_layers):
    layer_proj = df['per_layer_projections'].apply(lambda x: x[L])
    rho_L, p_L = stats.spearmanr(layer_proj, df['A_displaced'])
    layer_rhos.append(rho_L)
    if p_L < 0.05:
        print(f'  Layer {L:>2}: rho={rho_L:+.3f}  p={p_L:.4f}  *')
strongest_layer = int(np.argmax(np.abs(layer_rhos)))
print(f'  Strongest layer: L{strongest_layer}, rho={layer_rhos[strongest_layer]:+.3f}')

# Verdict
print('\n=== VERDICT ===')
if rho_proj < -0.4 and p_proj < 0.05:
    print('  >>> HYPOTHESIS SUPPORTED: B opposes A in residual stream → more damage.')
    print('  >>> NOVEL: damage is residual-direction-aligned. Mechanistic localization confirmed.')
elif rho_proj > 0.4 and p_proj < 0.05:
    print('  >>> REVERSED: B reinforcing A in residual space → more damage. Counterintuitive.')
elif p_proj < 0.05 or p_cos < 0.05:
    print(f'  >>> Significant but moderate effect. proj rho={rho_proj:+.3f}, cos rho={rho_cos:+.3f}')
else:
    print('  >>> Hypothesis NOT supported. Residual-direction metrics do not predict damage.')
    print('  >>> Damage may operate at finer granularity (per-token, per-position, sub-layer).')

### 18.7 Money plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.scatter(df['projection_signed'], df['A_displaced'], s=80, alpha=0.7,
           color='#cc3333', edgecolors='black', lw=0.4)
for _, r in df.iterrows():
    ax.annotate(f"{int(r['pair_idx'])}", (r['projection_signed'], r['A_displaced']),
                fontsize=8, alpha=0.7, ha='center')
ax.set_xlabel('Signed projection: ⟨δh_B, h_edit_A⟩ / |h_edit_A|')
ax.set_ylabel('A_displaced')
ax.set_title(f'B perturbation along A direction\nSpearman rho={rho_proj:+.3f}, p={p_proj:.4f}')
ax.grid(alpha=0.3); ax.axhline(0, color='black', lw=0.5); ax.axvline(0, color='black', lw=0.5)

ax = axes[1]
ax.scatter(df['cos_perturbation_vs_edit'], df['A_displaced'], s=80, alpha=0.7,
           color='#3366cc', edgecolors='black', lw=0.4)
ax.set_xlabel('cosine(δh_B, h_edit_A) in residual space')
ax.set_ylabel('A_displaced')
ax.set_title(f'Cosine in residual space\nSpearman rho={rho_cos:+.3f}, p={p_cos:.4f}')
ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(range(n_layers), layer_rhos, '-o', color='#33aa33', ms=6)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Layer index')
ax.set_ylabel('Spearman rho (per-layer projection vs A_displaced)')
ax.set_title('Which layer carries the damage signal?')
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_residual_direction.png', dpi=140)
plt.show()

### 18.8 Save

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 18 — Residual Stream Direction Hypothesis',
    'timestamp': ts, 'model': MODEL_NAME, 'n_pairs': len(pairs),
    'data_source': 'pair indices auto-fetched from github.com/rukmini-17/targeted-unlearning',
    'hypothesis': 'B opposes A in residual stream direction → more A_displaced',
    'n_clean_rows': len(df),
    'projection_signed_vs_A_displaced': {
        'spearman_rho': float(rho_proj), 'spearman_p': float(p_proj),
    },
    'cosine_in_residual_vs_A_displaced': {
        'spearman_rho': float(rho_cos), 'spearman_p': float(p_cos),
    },
    'magnitude_delta_h_vs_A_displaced': {
        'spearman_rho': float(rho_mag), 'spearman_p': float(p_mag),
    },
    'multivariate_R2': float(r2),
    'per_layer_strongest': {
        'layer': int(strongest_layer),
        'rho': float(layer_rhos[strongest_layer]),
    },
    'mean_projection_signed': float(df['projection_signed'].mean()),
    'verdict': ('hypothesis_supported_negative' if (rho_proj < -0.4 and p_proj < 0.05)
                else ('hypothesis_reversed' if (rho_proj > 0.4 and p_proj < 0.05)
                      else ('moderate_effect' if (p_proj < 0.05 or p_cos < 0.05)
                            else 'hypothesis_not_supported'))),
}
with open(RESULTS_DIR / f'summary_nb18_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
df_save = df.drop(columns=['per_layer_projections']).copy()
df_save.to_csv(RESULTS_DIR / f'df_nb18_{ts}.csv', index=False)
# Save per-layer too
per_layer_df = pd.DataFrame({
    'pair_idx': df['pair_idx'].values,
    **{f'layer_{L}': df['per_layer_projections'].apply(lambda x, L=L: x[L]) for L in range(n_layers)}
})
per_layer_df.to_csv(RESULTS_DIR / f'per_layer_projections_{ts}.csv', index=False)

print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb18_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')